In [1]:
import pandas as pd
from six_nations_fantasy.data import load_dataset
from six_nations_fantasy.models import PositionShrinkageMean

In [2]:
data = load_dataset()
pm = data.player_matches()
matches = data.matches
players = pd.read_parquet("../data/rebuild/players.parquet")

In [3]:
def predicted_skill_column_name(stat):
    return f"skill({stat})"

def skill(training, fixtures, stat):
    model = PositionShrinkageMean(k=20, scorer=lambda x: x[stat])
    model.fit(training)

    predicted = model.predict(fixtures)

    # the baseline is the global average at the players upcoming position & role
    baseline = model.baselines(fixtures)

    relative_strength = (predicted / baseline)#.rename(predicted_skill_column_name(stat))
    skill_rating = 1 - (fixtures[stat] - predicted)**2 / (fixtures[stat] - baseline)**2
    
    return pd.DataFrame({"relative_strength": relative_strength, "skill_rating": skill_rating})

In [4]:
start_date = pd.to_datetime("2026-02-05")
end_date = pd.to_datetime("2026-03-14")

training = data.select_matches(matches.loc[matches["date"] < start_date, "match_id"])
test = pm[(pm["date"].between(start_date, end_date, inclusive="both")) & (pm["competition_id"] == 0)]

In [5]:
# The m most skilled players (relative to their position) at the given stat 
# for the upcoming fixtures
def most_skilled(stat, m=5):
    relative_strength = skill(training, test, stat)["relative_strength"]
    
    return (pd.concat([relative_strength, test["player_id"]], axis=1)
        .groupby("player_id")
        .agg(predicted=("relative_strength", "mean"))
        .nlargest(m, columns="predicted")
        .join(players["name"], on="player_id")
    )

In [6]:
def all_ratings(method="relative_strength"):
    # Now find the most skilled players in general (the players with the highest skill in any particular stat)
    selected_stats = ['conversions', 'penalty_goals', 'drop_goals', 'carries', 'clean_breaks',
           'completed_tackles', 'turnovers_conceded', 'turnovers_won',
           'total_kicks', 'passes', 'carried_meters', 'offloads',
           'defenders_beaten', 'try_assists', 'tries', 'carries_per_minute',
           'total_tackles', 'dominant_tackles', 'ruck_turnovers',
           'lineouts_won', 'tackles_per_minute']
    
    return pd.concat([skill(training, test, stat)[method].rename(stat) for stat in selected_stats] + [test["player_id"]], axis=1)

In [15]:
all_skills = all_ratings(method="skill_rating")
(all_skills
    .set_index("player_id")
    .agg(["idxmax", "max"], axis=1)
    .astype({"max": "float64"})
    #.groupby("player_id")
    #.agg(
    #    best_skill=("idxmax", "first"), # idxmin should be constant over the group
    #    value=("max", "max")) # for rating players against each other, keep their best rating in their best skill (different ratings can occur in different roles)
    .nlargest(10, columns="max")
    .join(players["name"], on="player_id")
)

,idxmax,max,name
player_id,,,
137626,total_tackles,1.000000,Finlay Smith
137493,carries,0.999997,Antoine Dupont
157863,carries,0.999996,Finn Russell
165007,carries,0.999993,Ellis Genge
224426,tackles_per_minute,0.999978,Tom O'Toole
177559,carried_meters,0.999950,Alex Mitchell
182170,passes,0.999943,Ryan Elias
182170,passes,0.999943,Ryan Elias
224426,carries,0.999942,Tom O'Toole


TODO: 
- formulate exactly what the two different "method"s measure, as the results are a little different
- the "skill" rating shows what we can best predict(?)... These are kicking stats, lineouts_won, passes, carries, ruck_turnovers, carried_meters (again, quite reasonable)

In [16]:
(all_skills
    .quantile(q=0.9, axis=0)
    .drop("player_id")
    .sort_values(ascending=False)
)

drop_goals            0.940512
penalty_goals         0.937500
conversions           0.930748
lineouts_won          0.928889
passes                0.898862
carries               0.874639
ruck_turnovers        0.868680
total_kicks           0.867769
carried_meters        0.842358
try_assists           0.833403
carries_per_minute    0.824698
tackles_per_minute    0.777370
completed_tackles     0.765509
defenders_beaten      0.762434
total_tackles         0.729308
clean_breaks          0.707816
tries                 0.691597
offloads              0.680783
turnovers_won         0.563024
dominant_tackles      0.554289
turnovers_conceded    0.494574
Name: 0.9, dtype: float64

In [9]:
most_skilled("try_assists")

,predicted,name
player_id,,
165007,6.883017,Ellis Genge
138386,6.134717,Baptiste Serin
218331,4.214606,Jarrod Evans
176354,3.814163,Nathan Doak
113737,3.616363,Jack Van Poortvliet
